In [1]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# 1. Load the dataset
# Ensure 'autism_screening.csv' is in the same folder as this script
df = pd.read_csv('autism_screening.csv')

# 2. Data Cleaning
df['ethnicity'] = df['ethnicity'].replace('?', 'Others')
df.loc[df['age'] > 100, 'age'] = df['age'].median() # Fix outliers
df['age'] = df['age'].fillna(df['age'].median())

# 3. Feature Selection (Matching the UI)
features = ['A1_Score', 'A2_Score', 'A3_Score', 'A4_Score', 'A5_Score',
            'A6_Score', 'A7_Score', 'A8_Score', 'A9_Score', 'A10_Score',
            'age', 'gender', 'jundice', 'austim']

X = df[features].copy()
y = df['Class/ASD'].apply(lambda x: 1 if x == 'YES' else 0)

# 4. Encoding Categorical Variables
# Gender: f=0, m=1 | Jaundice: no=0, yes=1 | Autism history: no=0, yes=1
le = LabelEncoder()
X['gender'] = le.fit_transform(X['gender'])
X['jundice'] = le.fit_transform(X['jundice'])
X['austim'] = le.fit_transform(X['austim'])

# 5. Train the Model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 6. Save the trained model
with open('autism_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Success: 'autism_model.pkl' has been created.")

Success: 'autism_model.pkl' has been created.


In [ ]:
import tkinter as tk
from tkinter import messagebox, ttk
import pickle
import numpy as np

# Load the trained model
try:
    with open('autism_model.pkl', 'rb') as f:
        model = pickle.load(f)
except FileNotFoundError:
    print("Error: 'autism_model.pkl' not found. Run the trainer script first!")

# Your custom AQ-10 questions
aq_questions = [
    ('A1' , 'I often notice small sounds when others do not.'),
    ('A2' , 'I usually concentrate more on the whole picture, rather than small details.'),
    ('A3' , 'I find it easy to do more than one thing at once.'),
    ('A4' , 'If there is an interruption, I can switch back quickly to what I was doing.'),
    ('A5' , 'I find it easy to read between the lines when someone talks to me.'),
    ('A6' , 'I know how to tell if someone listening to me is getting bored.'),
    ('A7' , 'When reading a story, I find it difficult to work out the characters intentions.'),
    ('A8' , 'I like to collect information about categories of things such as cars or birds.'),
    ('A9' , 'I find it easy to work out what someone is thinking or feeling just by looking.'),
    ('A10', 'I find it difficult to work out what other people are intending to do.'),
]

def predict_asd():
    try:
        # 1. Collect A1-A10 answers (0 or 1)
        inputs = [int(var.get()) for var in score_vars]
        
        # 2. Collect Age, Gender, Jaundice, and Family History
        age = float(age_entry.get())
        gender = 1 if gender_var.get() == "Male" else 0
        jaundice = 1 if jaundice_var.get() == "Yes" else 0
        family_history = 1 if family_var.get() == "Yes" else 0
        
        inputs.extend([age, gender, jaundice, family_history])

        # 3. Prediction logic
        features = np.array([inputs])
        prediction = model.predict(features)

        # 4. Result Popup
        if prediction[0] == 1:
            messagebox.showwarning("Result", "The screening indicates potential ASD traits. Please consult a professional.")
        else:
            messagebox.showinfo("Result", "The screening indicates no significant ASD traits.")
            
    except ValueError:
        messagebox.showerror("Error", "Please enter a valid numeric age.")

# --- UI Setup ---
root = tk.Tk()
root.title("Autism Screening Tool (AQ-10)")
root.geometry("700x700")

# Create a Scrollable Frame
canvas = tk.Canvas(root)
scrollbar = ttk.Scrollbar(root, orient="vertical", command=canvas.yview)
scrollable_frame = tk.Frame(canvas)

scrollable_frame.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))
canvas.create_window((0, 0), window=scrollable_frame, anchor="nw")
canvas.configure(yscrollcommand=scrollbar.set)

canvas.pack(side="left", fill="both", expand=True)
scrollbar.pack(side="right", fill="y")

# Title
tk.Label(scrollable_frame, text="AQ-10 Adult Screening", font=("Arial", 16, "bold"), pady=10).pack()

# Question Rendering
score_vars = []
for code, text in aq_questions:
    q_frame = tk.LabelFrame(scrollable_frame, text=code, padx=10, pady=5)
    q_frame.pack(fill="x", padx=20, pady=5)
    
    tk.Label(q_frame, text=text, wraplength=500, justify="left").pack(side="left")
    
    var = tk.StringVar(value="0")
    score_vars.append(var)
    tk.Radiobutton(q_frame, text="Yes", variable=var, value="1").pack(side="right")
    tk.Radiobutton(q_frame, text="No", variable=var, value="0").pack(side="right")

# Personal Details
detail_frame = tk.LabelFrame(scrollable_frame, text="Personal Details", padx=10, pady=10)
detail_frame.pack(fill="x", padx=20, pady=10)

tk.Label(detail_frame, text="Age:").grid(row=0, column=0, sticky="w")
age_entry = tk.Entry(detail_frame)
age_entry.insert(0, "25")
age_entry.grid(row=0, column=1, padx=10)

tk.Label(detail_frame, text="Gender:").grid(row=1, column=0, sticky="w")
gender_var = tk.StringVar(value="Male")
ttk.Combobox(detail_frame, textvariable=gender_var, values=["Male", "Female"]).grid(row=1, column=1, padx=10)

tk.Label(detail_frame, text="Born with Jaundice?").grid(row=2, column=0, sticky="w")
jaundice_var = tk.StringVar(value="No")
ttk.Combobox(detail_frame, textvariable=jaundice_var, values=["Yes", "No"]).grid(row=2, column=1, padx=10)

tk.Label(detail_frame, text="Family History of Autism?").grid(row=3, column=0, sticky="w")
family_var = tk.StringVar(value="No")
ttk.Combobox(detail_frame, textvariable=family_var, values=["Yes", "No"]).grid(row=3, column=1, padx=10)

# Predict Button
tk.Button(scrollable_frame, text="GET PREDICTION", command=predict_asd, 
          bg="#2ecc71", fg="white", font=("Arial", 12, "bold"), pady=10).pack(pady=20)

root.mainloop()